In [ ]:
"""
Initializes visualization settings, loads the dataset, cleans column names,
and filters for valid target classes.
"""
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/data.csv', sep=';', engine='python')
df.columns = [c.strip().strip('"').replace("\t", "").strip() for c in df.columns]
df = df[df["Target"].isin(["Graduate", "Dropout"])].copy()

df.head()

Dataset overview and missing-value check.

In [ ]:
"""
Displays basic dataset information and counts total missing values.
"""
print(df.info())
print("\n" + "="*40)
print(f"Total Missing Values in Dataset: {df.isnull().sum().sum()}")

> **Insight 1: Data Quality & Completeness**
> The dataset consists of 3,630 records and 37 features. Notably, there are zero missing values. This eliminates the need for complex imputation techniques and ensures high-quality data for model training.

Target-class distribution, to see whether the classes are imbalanced.

In [ ]:
"""
Plots the distribution of the target classes and annotates each bar
with its corresponding percentage for better readability.
"""
plt.figure(figsize=(8, 5))

# Pass the same column to hue and disable the legend to avoid a seaborn warning.
ax = sns.countplot(data=df, x='Target', hue='Target', palette='Set2', legend=False)

plt.title('Target Class Distribution: Graduate vs Dropout', fontweight='bold', pad=15)

total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2 - 0.1
    y = p.get_y() + p.get_height() + 10
    ax.annotate(percentage, (x, y), fontweight='bold')

plt.tight_layout()
plt.show()

> **Insight 2: Class Distribution & Imbalance**
> The target variable exhibits a moderate class imbalance (60.9% Graduate vs. 39.1% Dropout). While this doesn't necessitate aggressive sampling techniques like SMOTE, it justifies the use of `scale_pos_weight` in XGBoost and `Stratified K-Fold` cross-validation to prevent model bias toward the majority class.

Pearson correlation among numeric features, to spot linear redundancy.

In [ ]:
"""
Extracts numeric features, calculates the Pearson correlation matrix,
and visualizes it using a heatmap without annotations to prevent clutter.
"""
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(16, 12))
corr = df[numeric_cols].corr()

sns.heatmap(
    corr, 
    cmap='coolwarm', 
    center=0, 
    annot=False, 
    linewidths=0.5
)

plt.title('Correlation Matrix of Numeric Features', fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

> **Insight 3: Multicollinearity & Feature Relationships**
> The heatmap reveals strong positive linear correlations among academic performance metrics across semesters (e.g., enrolled units, approved units, and grades). This severe multicollinearity directly validates our rigorous Feature Selection pipeline (combining Mutual Information and L1 Regularization) to eliminate redundant features and mitigate the risk of overfitting.